In [1]:
import warnings 
warnings.filterwarnings("ignore")

import os
import inspect
import traceback
import time
from pathlib import Path

from collections import Counter
import missingno as ms
import sweetviz as sv

import pandas as pd
import numpy as np

In [2]:
ROOT_DIR = Path().cwd().parents[0]
os.chdir(ROOT_DIR)

from src.config import Load_data, Database, Reports
os.chdir(ROOT_DIR/Load_data)
from load_data import read_data
os.chdir(ROOT_DIR/Database)
from db_connection import engine

C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project


2026-09-19 23:37:36,508 - INFO - loading env varailbles...
2026-09-19 23:37:36,509 - INFO - env variables loaded and saved!
2026-09-19 23:37:36,510 - INFO - intiating connection string..
2026-09-19 23:37:36,512 - INFO - intiated connection string!
2026-09-19 23:37:36,512 - INFO - creating alchemy Engine...
2026-09-19 23:37:36,589 - INFO - Engine Created Successfully.


In [3]:
pd.set_option("display.float_format", "{:.2f}".format)
np.set_printoptions(precision = 2, suppress = True)

In [4]:
inspect.signature(read_data)

<Signature (tablename: str) -> None>

In [5]:
customers_data = read_data("customers")

2026-09-19 23:37:37,723 - INFO - Changing directiory...
2026-09-19 23:37:37,729 - INFO - loading dataset...
2026-09-19 23:37:37,852 - INFO - @@@@ dataset Readed Successfuly. @@@


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw
Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    220.0 MiB    220.0 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    220.0 MiB      0.0 MiB           1       try:
    19    220.0 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    220.0 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    220.0 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    220.0 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    227.0 MiB      7.0 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    227.0 MiB      0.0 MiB           1 

In [6]:
print("", end = "\n")
display(f"Shape of Dataset : {customers_data.shape}")
print("", end = "\n")
display(customers_data.sample(n = 5))

'Shape of Dataset : (10000, 17)'

,customer_id,first_name,last_name,email,phone,city,state,pincode,gender,age,segment,registration_date,last_login_date,is_verified,is_active,total_orders,total_spent
707,CUST00708,Meera,Srivastava,meera.srivastava744@gmail.com,9522818091,Chennai,Tamil Nadu,201972,Female,42,Budget,19-12-2020,28-04-2025,1,1,7,431879.14
4288,CUST04289,Deepak,Mukherjee,deepak.mukherjee855@gmail.com,9266665224,Jodhpur,Rajasthan,272195,Female,57,New,25-12-2020,10-11-2023,1,1,5,79802.36
7828,CUST07829,Ishaan,Joshi,ishaan.joshi313@gmail.com,9695781668,Solapur,Maharashtra,396961,Male,41,Regular,29-04-2022,28-02-2024,1,1,6,877874.80
8298,CUST08299,Akash,Das,akash.das90@yahoo.in,9761309457,Kanpur,Uttar Pradesh,663468,Male,65,Premium,08-04-2025,09-04-2025,1,1,6,98864.67
7222,CUST07223,Varun,Reddy,varun.reddy70@gmail.com,9422894029,Visakhapatnam,Andhra Pradesh,168528,Female,35,Budget,29-12-2020,09-06-2024,1,1,2,336011.96


In [7]:
display(customers_data[customers_data["customer_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated customer id's : {len(customers_data[customers_data["customer_id"].duplicated()])}")

,customer_id,first_name,last_name,email,phone,city,state,pincode,gender,age,segment,registration_date,last_login_date,is_verified,is_active,total_orders,total_spent


"Total Number of duplicated customer id's : 0"

In [8]:
display(customers_data.isnull().sum())
print("", end = "\n")

for col, nv in enumerate(customers_data.isnull().sum()):
    if nv > 0:
        print(f"Column {col}: no of nulls : {nv}")

customer_id          0
first_name           0
last_name            0
email                0
phone                0
city                 0
state                0
pincode              0
gender               0
age                  0
segment              0
registration_date    0
last_login_date      0
is_verified          0
is_active            0
total_orders         0
total_spent          0
dtype: int64

In [9]:
customers_data.dtypes

customer_id           object
first_name            object
last_name             object
email                 object
phone                  int64
city                  object
state                 object
pincode                int64
gender                object
age                    int64
segment               object
registration_date     object
last_login_date       object
is_verified            int64
is_active              int64
total_orders           int64
total_spent          float64
dtype: object

In [10]:
customers_data["registration_date"] = pd.to_datetime(customers_data["registration_date"]).dt.date
customers_data["last_login_date"] = pd.to_datetime(customers_data["last_login_date"], format = "mixed").dt.date

In [11]:
display(customers_data[customers_data["total_orders"] == 0].head())

if (customers_data["total_orders"] == 0).any():
    print("", end = "\n")
    display(f"Total customers : {len(customers_data[customers_data["total_orders"] == 0])}")

,customer_id,first_name,last_name,email,phone,city,state,pincode,gender,age,segment,registration_date,last_login_date,is_verified,is_active,total_orders,total_spent
65,CUST00066,Geeta,Mehta,geeta.mehta895@gmail.com,9652228587,Moradabad,Uttar Pradesh,389882,Male,44,New,2022-04-10,2023-02-03,1,1,0,0.00
121,CUST00122,Ravi,Ghosh,ravi.ghosh911@gmail.com,9334559782,Madurai,Tamil Nadu,444600,Male,25,Inactive,2020-08-15,2020-10-21,1,1,0,0.00
123,CUST00124,Ashish,Pandey,ashish.pandey103@gmail.com,9986400277,Navi Mumbai,Maharashtra,690717,Female,62,Inactive,2022-01-05,2023-02-07,1,1,0,0.00
127,CUST00128,Sanjay,Dwivedi,sanjay.dwivedi962@yahoo.in,9683627205,Solapur,Maharashtra,500721,Male,38,Inactive,2021-01-06,2025-07-06,0,0,0,0.00
161,CUST00162,Ashish,Ghosh,ashish.ghosh439@gmail.com,9470012350,Kolkata,West Bengal,285383,Female,59,Inactive,2022-11-17,2025-05-23,1,1,0,0.00


'Total customers : 457'

In [12]:
orders_data = read_data("orders")

2026-09-19 23:37:38,155 - INFO - Changing directiory...
2026-09-19 23:37:38,160 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:38,587 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    228.7 MiB    228.7 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    228.7 MiB      0.0 MiB           1       try:
    19    228.7 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    228.7 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    228.7 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    228.7 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    239.9 MiB     11.2 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    239.9 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    239.9 MiB      0.0 MiB           1          

In [13]:
display(orders_data.head())
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(orders_data.info())

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
0,ORD000001,CUST02946,22-05-2025,17:10:00,Processing,Pune,Maharashtra,300862,100984.19,16364.64,0,20486.45,100984.19,Cash on Delivery,BlueDart,TRK354572763,NaN,1,App
1,ORD000002,CUST04232,16-10-2023,05:32:00,Delivered,Ludhiana,Punjab,387852,254478.01,41599.80,0,19338.79,254478.01,UPI,DTDC,TRK272838570,19-10-2023,0,Website
2,ORD000003,CUST08078,08-10-2023,12:32:00,Cancelled,Meerut,Uttar Pradesh,294014,32557.06,3562.92,0,696.86,32557.06,Debit Card,Amazon Logistics,TRK338505401,NaN,0,App
3,ORD000004,CUST09612,02-10-2023,08:05:00,Delivered,Nashik,Maharashtra,626047,24836.94,3092.88,0,4029.94,24836.94,UPI,DTDC,TRK289792017,09-10-2023,0,Mobile Web
4,ORD000005,CUST03065,15-06-2024,20:19:00,Delivered,Moradabad,Uttar Pradesh,409210,156689.29,25092.36,0,7805.07,156689.29,UPI,Ecom Express,TRK978240041,22-06-2024,0,App


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          50000 non-null  object 
 1   customer_id       50000 non-null  object 
 2   order_date        50000 non-null  object 
 3   order_time        50000 non-null  object 
 4   status            50000 non-null  object 
 5   city              50000 non-null  object 
 6   state             50000 non-null  object 
 7   pincode           50000 non-null  int64  
 8   total_amount      50000 non-null  float64
 9   gst_amount        50000 non-null  float64
 10  shipping_charge   50000 non-null  int64  
 11  discount_amount   50000 non-null  float64
 12  final_amount      50000 non-null  float64
 13  payment_method    50000 non-null  object 
 14  shipping_partner  50000 non-null  object 
 15  tracking_id       50000 non-null  object 
 16  delivered_date    32499 non-null  objec

None

In [14]:
display(orders_data[orders_data["order_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated customer id's : {len(orders_data[orders_data["order_id"].duplicated()])}")

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel


"Total Number of duplicated customer id's : 0"

In [15]:
display(orders_data.isnull().sum())
print("", end = "\n")

for idx, (col,nv) in enumerate(orders_data.isnull().sum().items()):
    if nv > 0:
        print(f"Column {col}: no of nulls : {nv} : percentage : {round(100 * nv/len(orders_data), 2)}")

order_id                0
customer_id             0
order_date              0
order_time              0
status                  0
city                    0
state                   0
pincode                 0
total_amount            0
gst_amount              0
shipping_charge         0
discount_amount         0
final_amount            0
payment_method          0
shipping_partner        0
tracking_id             0
delivered_date      17501
is_cod                  0
channel                 0
dtype: int64


Column delivered_date: no of nulls : 17501 : percentage : 35.0


In [16]:
orders_data["order_date"] = pd.to_datetime(orders_data["order_date"]).dt.date
orders_data["delivered_date"] = pd.to_datetime(orders_data["delivered_date"], format = "mixed").dt.date

In [17]:
display(orders_data[orders_data["final_amount"] > 500000].head())
print("", end = "\n")
display(f"total outliers in Final Amount of orders : {len(orders_data[orders_data["final_amount"] > 500000])}")

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
33,ORD000034,CUST01867,2024-03-10,07:18:00,Delivered,Gwalior,Madhya Pradesh,714121,527178.05,87982.56,0,85216.51,527178.05,Credit Card,XpressBees,TRK426129266,2024-03-17,0,App
34,ORD000035,CUST01370,2025-05-26,17:56:00,Cancelled,Mumbai,Maharashtra,135015,532061.13,82016.52,0,11940.39,532061.13,UPI,DTDC,TRK731489123,NaT,0,App
306,ORD000307,CUST03874,2024-03-20,11:55:00,Cancelled,Raipur,Chhattisgarh,721504,550716.17,85706.88,0,40438.71,550716.17,Cash on Delivery,Delhivery,TRK499571364,NaT,1,App
583,ORD000584,CUST07624,2024-10-30,02:55:00,Delivered,Hyderabad,Telangana,487237,745667.68,126685.30,0,81057.62,745667.68,Cash on Delivery,BlueDart,TRK820123719,2024-04-11,1,Website
590,ORD000591,CUST05298,2025-05-22,04:28:00,Cancelled,Faridabad,Haryana,262336,520293.72,84126.96,0,31732.24,520293.72,Credit Card,Shadowfax,TRK957301189,NaT,0,App


'total outliers in Final Amount of orders : 312'

In [18]:
display(orders_data[orders_data["order_date"] > orders_data["delivered_date"]].head())
print("", end = "\n")
display(f"total invalid dates : {len(orders_data[orders_data["order_date"] > orders_data["delivered_date"]])}")

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
3,ORD000004,CUST09612,2023-10-02,08:05:00,Delivered,Nashik,Maharashtra,626047,24836.94,3092.88,0,4029.94,24836.94,UPI,DTDC,TRK289792017,2023-09-10,0,Mobile Web
18,ORD000019,CUST03475,2025-01-25,04:54:00,Delivered,Chennai,Tamil Nadu,757744,4252.56,536.22,0,522.66,4252.56,UPI,BlueDart,TRK598044502,2025-01-02,0,Mobile Web
25,ORD000026,CUST08711,2024-06-29,05:55:00,Delivered,Vijayawada,Andhra Pradesh,421801,142932.73,12407.52,0,8668.79,142932.73,UPI,XpressBees,TRK586755485,2024-03-07,0,Mobile Web
27,ORD000028,CUST06835,2023-06-26,22:45:00,Delivered,Meerut,Uttar Pradesh,683261,11875.41,1049.51,0,778.10,11875.41,Credit Card,Shadowfax,TRK331524583,2023-03-07,0,App
30,ORD000031,CUST07132,2023-10-02,02:52:00,Delivered,Aurangabad,Maharashtra,777152,73656.45,12495.00,0,6599.55,73656.45,UPI,Amazon Logistics,TRK941863167,2023-06-10,0,App


'total invalid dates : 5989'

In [19]:
order_items_data = read_data("order_items")

2026-09-19 23:37:39,511 - INFO - Changing directiory...
2026-09-19 23:37:39,514 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:39,915 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    245.5 MiB    245.5 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    245.5 MiB      0.0 MiB           1       try:
    19    245.5 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    245.5 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    245.5 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    245.5 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    261.3 MiB     15.8 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    261.3 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    261.3 MiB      0.0 MiB           1          

In [20]:
display(order_items_data.head())
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(order_items_data.info())

,item_id,order_id,product_id,product_name,category,quantity,unit_price,gst_rate,gst_amount,discount_amount,total_price
0,ITM0000001,ORD000001,PRD0522,HCL Cricket Bat Ultra,Sports & Fitness,1,40989,12,4918.68,7861.45,38046.23
1,ITM0000002,ORD000001,PRD0094,Berger Router Pro,Electronics,1,62532,18,11255.76,12373.11,61414.65
2,ITM0000003,ORD000001,PRD0613,Voltas Educational Toy Standard,Toys & Baby,1,1585,12,190.20,251.89,1523.31
3,ITM0000004,ORD000002,PRD0556,Ambrane Cycle Standard,Sports & Fitness,3,1107,12,398.52,631.44,3088.08
4,ITM0000005,ORD000002,PRD0026,Pebble Tablet Premium,Electronics,2,114448,18,41201.28,18707.35,251389.93


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   item_id          100000 non-null  object 
 1   order_id         100000 non-null  object 
 2   product_id       100000 non-null  object 
 3   product_name     100000 non-null  object 
 4   category         100000 non-null  object 
 5   quantity         100000 non-null  int64  
 6   unit_price       100000 non-null  int64  
 7   gst_rate         100000 non-null  int64  
 8   gst_amount       100000 non-null  float64
 9   discount_amount  100000 non-null  float64
 10  total_price      100000 non-null  float64
dtypes: float64(3), int64(3), object(5)
memory usage: 8.4+ MB


None

In [21]:
display(order_items_data[order_items_data["item_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(order_items_data[order_items_data["item_id"].duplicated()])}")

,item_id,order_id,product_id,product_name,category,quantity,unit_price,gst_rate,gst_amount,discount_amount,total_price


'Total Number of duplicated item_id : 0'

In [22]:
display(order_items_data.isnull().sum())
print("", end = "\n")

for idx, (col,nv) in enumerate(order_items_data.isnull().sum().items()):
    if nv > 0:
        print(f"Column {col}: no of nulls : {nv} : percentage : {round(100 * nv/len(order_items_data), 2)}")

item_id            0
order_id           0
product_id         0
product_name       0
category           0
quantity           0
unit_price         0
gst_rate           0
gst_amount         0
discount_amount    0
total_price        0
dtype: int64

In [23]:
order_items_data["order_id"].isin(orders_data["order_id"]).all()

np.True_

In [24]:
suppliers_data = read_data("suppliers")

2026-09-19 23:37:40,231 - INFO - Changing directiory...
2026-09-19 23:37:40,238 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:40,296 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    263.8 MiB    263.8 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    263.8 MiB      0.0 MiB           1       try:
    19    263.8 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    263.8 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    263.8 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    263.8 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    264.2 MiB      0.4 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    264.2 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    264.2 MiB      0.0 MiB           1          

In [25]:
display(suppliers_data.head())
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(suppliers_data.info())

,supplier_id,supplier_name,contact_person,email,phone,city,state,pincode,category,gstin,payment_terms_days,rating,created_date,is_active
0,SUP0001,Iyer Elec Pvt Ltd,Harish Iyer,contact@iyerele.com,9362950628,Delhi,Delhi,334053,Electronics,14TRXCK1488C4Z9,15,4.10,22-05-2023,1
1,SUP0002,Sharma Offi Pvt Ltd,Mohit Sharma,contact@sharmaoff.com,9914763202,Ahmedabad,Gujarat,267414,Office Supplies,32KHFTC7224C6Z5,15,4.50,25-03-2023,1
2,SUP0003,Choudhury Beau Pvt Ltd,Nandini Choudhury,contact@choudhurybea.com,9764130526,Pune,Maharashtra,479201,Beauty & Health,28EBRUZ4814W7Z5,60,4.30,27-12-2022,1
3,SUP0004,Choudhury Fash Pvt Ltd,Vikram Choudhury,contact@choudhuryfas.com,9283758720,Chennai,Tamil Nadu,660086,Fashion,33GMHYR4598R1Z4,15,4.60,15-01-2023,1
4,SUP0005,Patil Spor Pvt Ltd,Kiran Patil,contact@patilspo.com,9636045484,Jaipur,Rajasthan,514850,Sports & Fitness,30LGGOG8019X7Z6,30,5.00,02-09-2022,1


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   supplier_id         200 non-null    object 
 1   supplier_name       200 non-null    object 
 2   contact_person      200 non-null    object 
 3   email               200 non-null    object 
 4   phone               200 non-null    int64  
 5   city                200 non-null    object 
 6   state               200 non-null    object 
 7   pincode             200 non-null    int64  
 8   category            200 non-null    object 
 9   gstin               200 non-null    object 
 10  payment_terms_days  200 non-null    int64  
 11  rating              200 non-null    float64
 12  created_date        200 non-null    object 
 13  is_active           200 non-null    int64  
dtypes: float64(1), int64(4), object(9)
memory usage: 22.0+ KB


None

In [26]:
display(suppliers_data[suppliers_data["supplier_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(suppliers_data[suppliers_data["supplier_id"].duplicated()])}")

,supplier_id,supplier_name,contact_person,email,phone,city,state,pincode,category,gstin,payment_terms_days,rating,created_date,is_active


'Total Number of duplicated item_id : 0'

In [27]:
display(suppliers_data.isnull().sum())
print("", end = "\n")

for idx, (col,nv) in enumerate(suppliers_data.isnull().sum().items()):
    if nv > 0:
        print(f"Column {col}: no of nulls : {nv} : percentage : {round(100 * nv/len(suppliers_data), 2)}")

supplier_id           0
supplier_name         0
contact_person        0
email                 0
phone                 0
city                  0
state                 0
pincode               0
category              0
gstin                 0
payment_terms_days    0
rating                0
created_date          0
is_active             0
dtype: int64

In [28]:
suppliers_data["created_date"] = pd.to_datetime(suppliers_data["created_date"]).dt.date

In [29]:
inventory_data = read_data("inventory")

2026-09-19 23:37:40,420 - INFO - Changing directiory...
2026-09-19 23:37:40,425 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:40,474 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    264.3 MiB    264.3 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    264.3 MiB      0.0 MiB           1       try:
    19    264.3 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    264.3 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    264.3 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    264.3 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    261.8 MiB     -2.5 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    261.8 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    261.8 MiB      0.0 MiB           1          

In [30]:
display(inventory_data.head())
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(inventory_data.info())

,inventory_id,product_id,warehouse_location,quantity_available,quantity_reserved,reorder_level,reorder_quantity,last_restocked_date,unit_cost,total_inventory_value,status
0,INV0001,PRD0001,Hyderabad-WH4,13,4,49,245,11-09-2023,55738,724594,Low Stock
1,INV0002,PRD0002,Kolkata-WH6,368,14,50,250,10-03-2025,48052,17683136,In Stock
2,INV0003,PRD0003,Delhi-WH2,196,19,44,220,22-11-2024,68913,13506948,In Stock
3,INV0004,PRD0004,Delhi-WH2,432,10,19,57,10-11-2024,80055,34583760,In Stock
4,INV0005,PRD0005,Bengaluru-WH3,404,16,44,88,05-08-2024,8398,3392792,In Stock


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   inventory_id           1000 non-null   object
 1   product_id             1000 non-null   object
 2   warehouse_location     1000 non-null   object
 3   quantity_available     1000 non-null   int64 
 4   quantity_reserved      1000 non-null   int64 
 5   reorder_level          1000 non-null   int64 
 6   reorder_quantity       1000 non-null   int64 
 7   last_restocked_date    1000 non-null   object
 8   unit_cost              1000 non-null   int64 
 9   total_inventory_value  1000 non-null   int64 
 10  status                 1000 non-null   object
dtypes: int64(6), object(5)
memory usage: 86.1+ KB


None

In [31]:
display(inventory_data[inventory_data["inventory_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(inventory_data[inventory_data["inventory_id"].duplicated()])}")

,inventory_id,product_id,warehouse_location,quantity_available,quantity_reserved,reorder_level,reorder_quantity,last_restocked_date,unit_cost,total_inventory_value,status


'Total Number of duplicated item_id : 0'

In [32]:
inventory_data["last_restocked_date"] = pd.to_datetime(inventory_data["last_restocked_date"], format = "mixed").dt.date

In [33]:
inventory_data["status"].unique()

array(['Low Stock', 'In Stock', 'Out of Stock'], dtype=object)

In [34]:
groupedby_status = inventory_data[inventory_data["status"] == "Out of Stock"].groupby("status")["product_id"].count()
groupedby_status

status
Out of Stock    2
Name: product_id, dtype: int64

In [35]:
products_data = read_data("products")

2026-09-19 23:37:40,615 - INFO - Changing directiory...
2026-09-19 23:37:40,622 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:40,686 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    261.9 MiB    261.9 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    261.9 MiB      0.0 MiB           1       try:
    19    261.9 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    261.9 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    261.9 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    261.9 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    264.4 MiB      2.4 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    264.4 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    264.4 MiB      0.0 MiB           1          

In [36]:
display(products_data.sample(n = 5))
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(products_data.info())

,product_id,product_name,category,subcategory,brand,sku,mrp,selling_price,cost_price,gst_rate,hsn_code,weight_grams,supplier_id,rating,review_count,is_active,launch_date
29,PRD0030,Infosys Tablet Elite,Electronics,Tablet,Infosys,SKU-ELE-00030,72918,58218,38045,18,1852,500,SUP0140,3.90,48,1,17-04-2022
837,PRD0838,HDFC Tyre Inflator Ultra,Automotive,Tyre Inflator,HDFC,SKU-AUT-00838,5568,4321,2802,28,4732,1500,SUP0099,4.80,2660,1,26-08-2024
51,PRD0052,Gizmore Bluetooth Speaker Ultra,Electronics,Bluetooth Speaker,Gizmore,SKU-ELE-00052,48054,42704,23800,18,5655,3000,SUP0157,4.30,1080,1,08-06-2022
659,PRD0660,Hindusta Unilever Puzzle Set Elite,Toys & Baby,Puzzle Set,Hindusta Unilever,SKU-TOY-00660,5501,4985,3421,12,9467,2000,SUP0052,4.00,1400,0,01-02-2025
795,PRD0796,Voltas Spice Kit Elite,Grocery,Spice Kit,Voltas,SKU-GRO-00796,151,133,82,0,3322,1500,SUP0137,4.60,3077,1,01-01-2025


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     1000 non-null   object 
 1   product_name   1000 non-null   object 
 2   category       1000 non-null   object 
 3   subcategory    1000 non-null   object 
 4   brand          1000 non-null   object 
 5   sku            1000 non-null   object 
 6   mrp            1000 non-null   int64  
 7   selling_price  1000 non-null   int64  
 8   cost_price     1000 non-null   int64  
 9   gst_rate       1000 non-null   int64  
 10  hsn_code       1000 non-null   int64  
 11  weight_grams   1000 non-null   int64  
 12  supplier_id    1000 non-null   object 
 13  rating         1000 non-null   float64
 14  review_count   1000 non-null   int64  
 15  is_active      1000 non-null   int64  
 16  launch_date    1000 non-null   object 
dtypes: float64(1), int64(8), object(8)
memory usage: 132

None

In [37]:
display(products_data[products_data["product_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(products_data[products_data["product_id"].duplicated()])}")

,product_id,product_name,category,subcategory,brand,sku,mrp,selling_price,cost_price,gst_rate,hsn_code,weight_grams,supplier_id,rating,review_count,is_active,launch_date


'Total Number of duplicated item_id : 0'

In [38]:
products_data["launch_date"] = pd.to_datetime(products_data["launch_date"], format = "mixed").dt.date
products_data["launch_date"]

0      2025-01-29
1      2024-05-17
2      2024-12-28
3      2023-04-05
4      2023-11-27
          ...    
995    2022-06-15
996    2024-07-17
997    2022-11-19
998    2024-04-28
999    2024-02-07
Name: launch_date, Length: 1000, dtype: object

In [39]:
returns_data = read_data("returns")

2026-09-19 23:37:40,822 - INFO - Changing directiory...


current cwd : 

2026-09-19 23:37:40,828 - INFO - loading dataset...


 C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:40,931 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    264.5 MiB    264.5 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    264.5 MiB      0.0 MiB           1       try:
    19    264.5 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    264.5 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    264.5 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    264.5 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    267.0 MiB      2.5 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    267.0 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    267.0 MiB      0.0 MiB           1          

In [40]:
display(returns_data.sample(n = 3))
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(returns_data.info())

,return_id,order_id,customer_id,return_date,reason,return_amount,refund_status,refund_date,return_condition,remarks
2222,RET02223,ORD028117,CUST09206,15-01-2024,Not as Described,31214.36,Refunded,19-01-2024,Damaged,Customer initiated return within 7 day window
2316,RET02317,ORD029201,CUST06597,03-01-2025,Quality Issue,413769.71,Refunded,10-01-2025,Damaged,Customer initiated return within 7 day window
4745,RET04746,ORD006728,CUST03634,26-01-2025,Size Issue,7686.75,Refunded,29-01-2025,Acceptable,Customer initiated return within 7 day window


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   return_id         10000 non-null  object 
 1   order_id          10000 non-null  object 
 2   customer_id       10000 non-null  object 
 3   return_date       10000 non-null  object 
 4   reason            10000 non-null  object 
 5   return_amount     10000 non-null  float64
 6   refund_status     10000 non-null  object 
 7   refund_date       10000 non-null  object 
 8   return_condition  10000 non-null  object 
 9   remarks           10000 non-null  object 
dtypes: float64(1), object(9)
memory usage: 781.4+ KB


None

In [41]:
display(returns_data[returns_data["return_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(returns_data[returns_data["return_id"].duplicated()])}")

,return_id,order_id,customer_id,return_date,reason,return_amount,refund_status,refund_date,return_condition,remarks


'Total Number of duplicated item_id : 0'

In [42]:
returns_data["return_date"] = pd.to_datetime(returns_data["return_date"], format = "mixed").dt.date
returns_data["refund_date"] = pd.to_datetime(returns_data["refund_date"], format = "mixed").dt.date

In [43]:
returns_data["order_id"].isin(orders_data["order_id"]).all()

np.True_

In [44]:
payments_data = read_data("payments")

2026-09-19 23:37:41,180 - INFO - Changing directiory...
2026-09-19 23:37:41,185 - INFO - loading dataset...


current cwd :  C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\data\raw


2026-09-19 23:37:41,579 - INFO - @@@@ dataset Readed Successfuly. @@@


Filename: C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\IndiaKart-E-Commerce-Analytics-Project\src\data\load_data.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    16    267.7 MiB    267.7 MiB           1   @profile
    17                                         def read_data(tablename: str)-> None:
    18    267.7 MiB      0.0 MiB           1       try:
    19    267.7 MiB      0.0 MiB           1           logging.info("Changing directiory...")
    20    267.7 MiB      0.0 MiB           1           os.chdir(ROOT_DIR/Raw_data)
    21    267.7 MiB      0.0 MiB           1           print("current cwd : ", os.getcwd())
    22    267.7 MiB      0.0 MiB           1           logging.info("loading dataset...")
    23    279.9 MiB     12.1 MiB           1           data = pd.read_csv(f"{tablename}.csv")
    24    279.9 MiB      0.0 MiB           1           logging.info("@@@@ dataset Readed Successfuly. @@@")
    25    279.9 MiB      0.0 MiB           1          

In [45]:
display(payments_data.sample(n = 5))
print("", end = "\n")
print("", end = "\n")
display("Basic Information of Dataset :")
print("", end = "\n")
display(payments_data.info())

,payment_id,order_id,customer_id,payment_date,payment_time,payment_method,amount,status,transaction_id,bank_name,gateway,refund_amount,refund_date
3105,PAY003106,ORD003106,CUST05124,18-11-2023,17:30:00,Net Banking,364295.82,Success,TXN6640951054,Kotak Mahindra,Razorpay,0,NaN
41798,PAY041799,ORD041799,CUST06131,12-10-2023,20:54:00,Cash on Delivery,244.83,Success,TXN3866552327,PNB,CCAvenue,0,NaN
42624,PAY042625,ORD042625,CUST06715,05-02-2024,08:47:00,UPI,494.48,Success,TXN8303346552,Axis Bank,Google Pay,0,NaN
46356,PAY046357,ORD046357,CUST08620,10-04-2024,16:15:00,UPI,23076.58,Success,TXN9386166358,Kotak Mahindra,Razorpay,0,NaN
5443,PAY005444,ORD005444,CUST03017,21-10-2023,18:12:00,Wallet,21970.90,Success,TXN9786490288,ICICI Bank,PhonePe,0,NaN


'Basic Information of Dataset :'


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   payment_id      50000 non-null  object 
 1   order_id        50000 non-null  object 
 2   customer_id     50000 non-null  object 
 3   payment_date    50000 non-null  object 
 4   payment_time    50000 non-null  object 
 5   payment_method  50000 non-null  object 
 6   amount          50000 non-null  float64
 7   status          50000 non-null  object 
 8   transaction_id  50000 non-null  object 
 9   bank_name       50000 non-null  object 
 10  gateway         50000 non-null  object 
 11  refund_amount   50000 non-null  int64  
 12  refund_date     0 non-null      float64
dtypes: float64(2), int64(1), object(10)
memory usage: 5.0+ MB


None

In [46]:
display(payments_data[payments_data["payment_id"].duplicated()])
print("", end = "\n")
print("", end = "\n")
display(f"Total Number of duplicated item_id : {len(payments_data[payments_data["payment_id"].duplicated()])}")

,payment_id,order_id,customer_id,payment_date,payment_time,payment_method,amount,status,transaction_id,bank_name,gateway,refund_amount,refund_date


'Total Number of duplicated item_id : 0'

In [47]:
payments_data["payment_date"] = pd.to_datetime(payments_data["payment_date"], format = "mixed").dt.date

In [48]:
display(Counter(payments_data["status"]))
failure_rate = 100 * dict(Counter(payments_data["status"]))["Failed"] / len(payments_data)
display(f"Failure Rate : {round(failure_rate,2)} %")

Counter({'Success': 48252, 'Failed': 1748})

'Failure Rate : 3.5 %'

In [49]:
datasets = [customers_data,orders_data,products_data,payments_data,order_items_data,inventory_data,suppliers_data,returns_data]
names = ["customers","orders","products", "payments", "order_items", "inventory", "suppliers", "returns"]
os.chdir(ROOT_DIR/Reports)
for idx, dataset in enumerate(datasets):
    report = sv.analyze(dataset)
    report.show_html(f"{names[idx]}_sweetviz_report.html")
print("Reports Saved Successfully.")

                                             |                                             | [  0%]   00:00 ->…

Report customers_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


                                             |                                             | [  0%]   00:00 ->…

Report orders_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


                                             |                                             | [  0%]   00:00 ->…

...

Report inventory_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


                                             |                                             | [  0%]   00:00 ->…

Report suppliers_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


                                             |                                             | [  0%]   00:00 ->…

Report returns_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
Reports Saved Successfully.


In [50]:
#customers_data = customers_data.drop_duplicates(subset=["email"],keep="first").reset_index(drop=True)

In [51]:
orders_data

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
0,ORD000001,CUST02946,2025-05-22,17:10:00,Processing,Pune,Maharashtra,300862,100984.19,16364.64,0,20486.45,100984.19,Cash on Delivery,BlueDart,TRK354572763,NaT,1,App
1,ORD000002,CUST04232,2023-10-16,05:32:00,Delivered,Ludhiana,Punjab,387852,254478.01,41599.80,0,19338.79,254478.01,UPI,DTDC,TRK272838570,2023-10-19,0,Website
2,ORD000003,CUST08078,2023-10-08,12:32:00,Cancelled,Meerut,Uttar Pradesh,294014,32557.06,3562.92,0,696.86,32557.06,Debit Card,Amazon Logistics,TRK338505401,NaT,0,App
3,ORD000004,CUST09612,2023-10-02,08:05:00,Delivered,Nashik,Maharashtra,626047,24836.94,3092.88,0,4029.94,24836.94,UPI,DTDC,TRK289792017,2023-09-10,0,Mobile Web
4,ORD000005,CUST03065,2024-06-15,20:19:00,Delivered,Moradabad,Uttar Pradesh,409210,156689.29,25092.36,0,7805.07,156689.29,UPI,Ecom Express,TRK978240041,2024-06-22,0,App
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,ORD049996,CUST06694,2023-11-30,02:02:00,Returned,Varanasi,Uttar Pradesh,211491,2922.08,745.36,0,485.28,2922.08,Credit Card,Delhivery,TRK377660669,NaT,0,Mobile Web
49996,ORD049997,CUST01649,2025-03-29,01:04:00,Cancelled,Nagpur,Maharashtra,475562,1844.05,312.48,0,689.71,1844.05,UPI,DTDC,TRK973715414,NaT,0,App
49997,ORD049998,CUST03865,2024-12-14,00:35:00,Delivered,Pune,Maharashtra,132579,13023.59,1452.96,0,1647.00,13023.59,Debit Card,Ecom Express,TRK187824914,2024-12-18,0,App
49998,ORD049999,CUST04997,2025-01-02,22:57:00,Shipped,Varanasi,Uttar Pradesh,782507,25068.84,4137.84,0,2057.00,25068.84,Credit Card,BlueDart,TRK205502040,NaT,0,Website


In [52]:
missing_customer_id_records = orders_data[~orders_data["customer_id"].isin(customers_data["customer_id"])]
display(f"mssing customer id's : {len(missing_customer_id_records)}")
display(f"total customer ids in customers : {len(customers_data)}")
display(f"total customer ids in orders : {len(customers_data)}")
display(missing_customer_id_records.head())

"mssing customer id's : 0"

'total customer ids in customers : 10000'

'total customer ids in orders : 10000'

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel


In [53]:
with engine.begin() as conn:
    conn.exec_driver_sql("DELETE FROM returns")
    conn.exec_driver_sql("DELETE FROM payments")
    conn.exec_driver_sql("DELETE FROM inventory")
    conn.exec_driver_sql("DELETE FROM order_items")
    conn.exec_driver_sql("DELETE FROM orders")
    conn.exec_driver_sql("DELETE FROM products")
    conn.exec_driver_sql("DELETE FROM suppliers")
    conn.exec_driver_sql("DELETE FROM customers")

customers_data.to_sql(
    name="customers",
    con=engine,
    if_exists="append",
    index=False
)

suppliers_data.to_sql(
    name="suppliers",
    con=engine,
    if_exists="append",
    index=False
)

products_data.to_sql(
    name="products",
    con=engine,
    if_exists="append",
    index=False
)

orders_data.to_sql(
    name="orders",
    con=engine,
    if_exists="append",
    index=False
)

order_items_data.to_sql(
    name="order_items",
    con=engine,
    if_exists="append",
    index=False
)

payments_data.to_sql(
    name="payments",
    con=engine,
    if_exists="append",
    index=False
)

inventory_data.to_sql(
    name="inventory",
    con=engine,
    if_exists="append",
    index=False
)

returns_data.to_sql(
    name="returns",
    con=engine,
    if_exists="append",
    index=False
)

print("All data loaded successfully.")

All data loaded successfully.
